# Design > Sampling

<div class="alert alert-info">Draw a simple random sample from a dataset</div>

The `sampling` function selects a simple random sample from a dataset. Each row in the dataset has an equal probability of being selected. A random number is assigned to each row, and the rows with the largest random numbers are selected. The random seed ensures reproducibility.

In [1]:
import polars as pl
import pyrsm as rsm

In [2]:
## setup pyrsm for autoreload
%reload_ext autoreload
%autoreload 2
%aimport pyrsm

# Example

We have a list of 100 names and want to draw a random sample of 10 for a survey. We load the `rndnames` dataset which contains 100 names with their gender.

In [3]:
rndnames = pl.read_parquet("../data/design/rndnames.parquet")
rndnames.head()

Names,Gender
str,enum
"""Ervin Escalona""","""Male"""
"""Allan Ammerman""","""Male"""
"""Milton Mothershed""","""Male"""
"""Deshawn Dawn""","""Male"""
"""Jc Julius""","""Male"""


In [4]:
s = rsm.design.sampling({"rndnames": rndnames}, vars=["Names"], sample_size=10, seed=1234)
s.summary()

Sampling (simple random)
Data       : rndnames
Variables  : Names
Random seed: 1234
Sample size: 10
Duplicates : Based on selected variables, no duplicate rows exist


The selected sample contains 10 randomly chosen names. Each name is accompanied by a `rnd_number` column showing the random number used for selection (higher numbers are selected first).

In [5]:
s.selected

Names,rnd_number
str,f64
"""Dominic Duhon""",0.992259
"""Antonina Alers""",0.988635
"""Foster Fugate""",0.977769
"""Ervin Escalona""",0.9767
"""Brooks Buie""",0.976417
"""Cyril Class""",0.964079
"""Dorinda Dehne""",0.956514
"""Gabriella Gurganus""",0.94736
"""Junko Jungers""",0.947199


# Using all variables

If you don't specify `vars`, all columns in the dataset are included in the sample:

In [6]:
s_all = rsm.design.sampling(rndnames, sample_size=10, seed=1234)
s_all.selected

Names,Gender,rnd_number
str,enum,f64
"""Dominic Duhon""","""Male""",0.992259
"""Antonina Alers""","""Female""",0.988635
"""Foster Fugate""","""Male""",0.977769
"""Ervin Escalona""","""Male""",0.9767
"""Brooks Buie""","""Male""",0.976417
"""Cyril Class""","""Male""",0.964079
"""Dorinda Dehne""","""Female""",0.956514
"""Gabriella Gurganus""","""Female""",0.94736
"""Junko Jungers""","""Female""",0.947199


# Reproducibility

Using the same seed produces the same sample every time. Changing the seed produces a different sample:

In [7]:
s1 = rsm.design.sampling(rndnames, vars=["Names"], sample_size=5, seed=42)
s2 = rsm.design.sampling(rndnames, vars=["Names"], sample_size=5, seed=42)
s3 = rsm.design.sampling(rndnames, vars=["Names"], sample_size=5, seed=99)

print("Same seed (42):")
print(f"  Sample 1: {s1.selected['Names'].to_list()}")
print(f"  Sample 2: {s2.selected['Names'].to_list()}")
print(f"\nDifferent seed (99):")
print(f"  Sample 3: {s3.selected['Names'].to_list()}")

Same seed (42):
  Sample 1: ['Denver Delph', 'Napoleon Norman', 'Hank Heavner', 'Denese Diem', 'Darrell Draheim']
  Sample 2: ['Denver Delph', 'Napoleon Norman', 'Hank Heavner', 'Denese Diem', 'Darrell Draheim']

Different seed (99):
  Sample 3: ['Bernadette Basnight', 'Linette Lavery', 'Deshawn Dawn', 'Lawrence Levasseur', 'Stacy Schrecengost']


© Vincent Nijs (2026)